Model

In [6]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, n_features=None, class_weights=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.class_weights = class_weights
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))

        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            thresholds = np.unique(X_col)

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left = y[X_col < threshold]
        right = y[X_col >= threshold]

        if len(left) == 0 or len(right) == 0:
            return float("inf")

        def weighted_gini(group):
            if len(group) == 0:
                return 0

            classes, counts = np.unique(group, return_counts=True)

            if self.class_weights is not None:
                weighted_counts = np.array([
                    counts[i] * self.class_weights.get(classes[i], 1)
                    for i in range(len(classes))
                ])
            else:
                weighted_counts = counts

            probs = weighted_counts / np.sum(weighted_counts)
            return 1 - np.sum(probs ** 2)

        n = len(y)
        return (len(left)/n)*weighted_gini(left) + (len(right)/n)*weighted_gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)

        if self.class_weights is not None:
            weighted_counts = np.array([
                counts[i] * self.class_weights.get(classes[i], 1)
                for i in range(len(classes))
            ])
        else:
            weighted_counts = counts

        return classes[np.argmax(weighted_counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

Training and Testing

In [7]:
import importlib
import preprocessing2
importlib.reload(preprocessing2)

<module 'preprocessing2' from 'c:\\~~~GAM3A~~~\\Semester 6\\machine learning\\project\\Image-classification-ML-Project\\phase_2\\preprocessing2.py'>

In [11]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca",n_pca=100)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 25.58 seconds.

  CUSTOM MODEL VALIDATION PERFORMANCE
                 precision     recall   f1-score    support

0                     0.92       0.90       0.91        587
1                     0.94       0.96       0.95        630
2                     0.74       0.80       0.77        600
3                     0.82       0.74       0.78        627
4                     0.87       0.74       0.80        595
5                     0.69       0.68       0.69        549
6                     0.85       0.86       0.85        571
7                     0.92       0.86       0.89        668
8                     0.67       0.78       0.72        597
9                     0.73       0.80       0.77        576

accuracy                                    0.81       6000
macro avg             0.82       0.81       0.81       6000

Valida

In [9]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 82.48 seconds.
                 precision     recall   f1-score    support

0                     0.84       0.82       0.83        587
1                     0.98       0.94       0.96        630
2                     0.66       0.85       0.74        600
3                     0.78       0.74       0.76        627
4                     0.88       0.83       0.86        595
5                     0.76       0.80       0.78        549
6                     0.89       0.83       0.86        571
7                     0.91       0.85       0.88        668
8                     0.71       0.73       0.72        597
9                     0.86       0.80       0.83        576

accuracy                                    0.82       6000
macro avg             0.83       0.82       0.82       6000

Validation Accuracy: 0.8198

Validation Confu

In [10]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 154.59 seconds.
                 precision     recall   f1-score    support

0                     0.93       0.93       0.93        587
1                     0.93       0.95       0.94        630
2                     0.83       0.91       0.87        600
3                     0.84       0.86       0.85        627
4                     0.87       0.86       0.87        595
5                     0.83       0.84       0.83        549
6                     0.94       0.90       0.92        571
7                     0.94       0.90       0.92        668
8                     0.87       0.80       0.83        597
9                     0.80       0.84       0.82        576

accuracy                                    0.88       6000
macro avg             0.88       0.88       0.88       6000

Validation Accuracy: 0.8783

Validation Conf